# Project FORESIGHT — 04: Production Forecasting, Inventory Risk Engine & Decision Support

**Objective**: Multi-horizon production forecast generation, inventory simulation under ratified supply chain policies, risk scoring, and operational decision support recommendations.

**Ratified Governance Policies (Milestone 5.X)**:
- **Decision #1 (Option 1D)**: Explicit Exclusion of Monetary Valuation. All financial/monetary metrics (`inventory_value_at_risk`, `excess_capital`) are strictly excluded and remain null.
- **Decision #2 (Option 2A)**: Policy B_LT On-Order Accounting. Verified supplier lead times $\le 14$ days ensure pending orders arrive within the active operational window.
- **Decision #3 (Option 3C)**: Overstock Threshold $N = 8$ Weeks ratified (`POLICY_RATIFIED`).
- **Universe**: 50 Production SKUs (`SKU001`–`SKU050`). 150 Orphan SKUs remain strictly quarantined.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG, PATHS
from src.production_pipeline import run_pipeline, ProductionPipeline

print(f"FORESIGHT Production Forecast & Risk Engine Initialized. Root: {PROJECT_ROOT}")

## 1. Multi-Horizon Production Demand Forecasts

Inspect the latest 8-week forward demand predictions generated by the production Hybrid model for all 50 production SKUs.

In [ ]:
pred_path = PATHS.artifacts_dir / "models" / "final" / "final_predictions.parquet"
if pred_path.exists():
    preds_df = pd.read_parquet(pred_path)
    latest_origin = preds_df["forecast_origin_date"].max()
    hybrid_preds = preds_df[(preds_df["forecast_origin_date"] == latest_origin) & 
                            (preds_df["model"].str.startswith("Selected Hybrid"))].copy()
    print(f"Loaded Production Forecasts for Origin Date: {pd.to_datetime(latest_origin).strftime('%Y-%m-%d')}")
    print(f"Total Forecast Records: {len(hybrid_preds)} (50 SKUs x 8 horizons)")
    print("\nSample Forecasts (First 5 SKUs):")
    sample_cols = ["SKU", "Category", "horizon", "forecast_date", "forecast_demand"]
    print(hybrid_preds[sample_cols].head(10).to_string(index=False))
else:
    print(f"Missing {pred_path}")

## 2. Inventory Simulation Under Policy B_LT (Decision #2 Option 2A)

Simulate projected inventory trajectories over the 8-week horizon accounting for on-hand stock, incoming purchase orders ($PO_{arrival} \le 14$ days), and forecasted demand consumption.

In [ ]:
inv_path = PATHS.raw_dir / "inventory.csv"
inv_df = pd.read_csv(inv_path)
prod_inv = inv_df[inv_df["SKU"].str.match(r"^SKU0[0-4][0-9]$|^SKU050$")].copy()

print(f"Active Fleet Inventory State (50 SKUs):")
print(f"  Total Current On-Hand Units : {prod_inv['Current_Stock'].sum():,}")
print(f"  Total Confirmed On-Order   : {prod_inv['On_Order'].sum():,}")
print(f"  Average Reorder Point (ROP): {prod_inv['Reorder_Point'].mean():.1f} units")
print(f"  Average Safety Stock       : {prod_inv['Safety_Stock'].mean():.1f} units")

## 3. Inventory Risk Engine & Stockout/Overstock Probabilities

Evaluate risk positions across the 50 production SKUs:
- **Stockout Risk**: Projected inventory breaches safety stock within supplier lead time.
- **Overstock Risk**: Projected inventory exceeds 8 weeks of forward demand (Decision #3 Option 3C).

In [ ]:
risk_path = PATHS.artifacts_dir / "risk" / "risk_scores_latest.parquet"
if risk_path.exists():
    risk_df = pd.read_parquet(risk_path)
    print("=== INVENTORY RISK PROFILING (50 SKUs) ===")
    if "risk_category" in risk_df.columns:
        print(risk_df["risk_category"].value_counts())
    elif "stockout_risk_level" in risk_df.columns:
        print(risk_df["stockout_risk_level"].value_counts())
    print("\nSample Risk Records:")
    cols_to_show = [c for c in ["SKU", "stockout_prob_leadtime", "days_of_supply", "is_overstocked", "risk_category"] if c in risk_df.columns]
    print(risk_df[cols_to_show].head(8).to_string(index=False))
else:
    print(f"Missing {risk_path}")

## 4. Operational Decision Support & Action Directives

Inspect the deterministic, prioritized replenishment recommendations:
- `REORDER`: Inventory below reorder point; order recommended quantity.
- `EXPEDITE`: Stockout imminent within lead time window; accelerate delivery.
- `OVERSTOCK_ALERT`: Excess stock exceeds 8-week threshold; throttle orders.
- `SURPLUS_TRANSFER`: Opportunity for rebalancing across distribution nodes.
- `MONITOR`: Healthy inventory within operating buffer.

In [ ]:
rec_path = PATHS.artifacts_dir / "phase6" / "production_latest_recommendations.parquet"
if rec_path.exists():
    recs_df = pd.read_parquet(rec_path)
    print("=== OPERATIONAL RECOMMENDATION DIRECTIVES ===")
    print(recs_df["recommendation_code"].value_counts())
    print("\nDirectives by Priority Rank (1=Highest, 5=Lowest):")
    print(recs_df["priority_rank"].value_counts().sort_index())
    print("\nHigh Priority Action Items (Priority <= 2):")
    high_pri = recs_df[recs_df["priority_rank"] <= 2]
    show_cols = ["sku", "category", "recommendation_code", "priority_rank", "recommended_order_units", "primary_rationale"]
    existing_cols = [c for c in show_cols if c in recs_df.columns]
    print(high_pri[existing_cols].head(6).to_string(index=False))
else:
    print(f"Missing {rec_path}")

## 5. Governance Compliance Audit (Decision #1 Option 1D)

Audit all production artifacts to guarantee strict compliance with ratified governance Decision #1 (Option 1D: Zero monetary valuation metrics exposed).

In [ ]:
if rec_path.exists():
    forbidden_monetary_cols = [
        "excess_inventory_value",
        "inventory_value_at_risk",
        "capital_at_risk",
        "cost_of_stockout",
        "carrying_cost_excess"
    ]
    violations = [col for col in forbidden_monetary_cols if col in recs_df.columns and recs_df[col].notna().any()]
    assert len(violations) == 0, f"GOVERNANCE VIOLATION: Exposed monetary metrics: {violations}"
    print("Governance Decision #1 (Option 1D): AUDIT PASSED [Zero monetary metrics exposed]")

## 6. Pipeline Manifest & Serving Verification

In [ ]:
manifest_path = PATHS.artifacts_dir / "phase6" / "pipeline_manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print("=== PRODUCTION PIPELINE RUN MANIFEST ===")
    print(f"Run ID        : {manifest.get('run_id')}")
    print(f"Status        : {manifest.get('status')}")
    print(f"Origin Date   : {manifest.get('actual_origin_date')}")
    print(f"SKUs Processed: {manifest.get('metrics', {}).get('production_skus_processed')}")
    print(f"Governance    : {manifest.get('governance_policies', {}).get('valuation_basis')}")
    print("Serving state : OPERATIONAL & READY")